In [ ]:
!pip install ARTSyn

In [ ]:
import pandas as pd
from artsyn.generators import ctd_gan
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
random_state = 42

In [ ]:
df1 = pd.read_csv("https://raw.githubusercontent.com/senihberkay/US-Army-ANSUR-II/master/ANSUR%20II%20FEMALE%20Public.csv", encoding='latin1')

In [ ]:
df2 = pd.read_csv("https://raw.githubusercontent.com/senihberkay/US-Army-ANSUR-II/master/ANSUR%20II%20MALE%20Public.csv", encoding='latin1')

In [ ]:
df1.head()

In [ ]:
df2.head()

In [ ]:
df2.rename(columns={'subjectid':'SubjectId'}, inplace=True)

In [ ]:
df1.columns == df2.columns

In [ ]:
print(df1.columns.tolist())

In [ ]:
df = pd.concat([df1, df2], ignore_index=True)
display(df.head())

In [ ]:
df.info()

In [ ]:
df.set_index('SubjectId', inplace=True)
display(df.head())

In [ ]:
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]
display(missing_values.sort_values(ascending=False))

In [ ]:
columns_to_drop = ['Date', 'Installation', 'Component', 'Branch', 'PrimaryMOS', 'SubjectsBirthLocation', 'SubjectNumericRace', 'Ethnicity', 'DODRace', 'Gender', 'Heightin', 'Weightlbs', 'WritingPreference']
df.drop(columns=columns_to_drop, inplace=True)
display(df.head())

In [ ]:
duplicate_rows = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

In [ ]:
df.describe()

In [ ]:
bins = [10, 19, 29, 39, 49, 59]
labels = ['10-19', '20-29', '30-39', '40-49', '50-59']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=True, include_lowest=True)

display(df[['Age', 'Age_Group']].head())

In [ ]:
df.drop(columns = "Age", inplace = True)

## Correlation Analysis for the Full Dataset (`df`)

In [ ]:


# Create a copy of df to avoid modifying the original and to add the encoded Age_Group
df_corr = df.copy()

# Fit the LabelEncoder on the 'Age_Group' column before transforming
le.fit(df_corr['Age_Group'])

# Encode the 'Age_Group' column using the already fitted LabelEncoder 'le'
df_corr['Age_Group_encoded'] = le.transform(df_corr['Age_Group'])

# Drop the original categorical 'Age_Group' column as we now have the encoded version
df_corr = df_corr.drop(columns=['Age_Group'])

# Calculate correlations with 'Age_Group_encoded'
df_correlations = df_corr.corr()['Age_Group_encoded'].drop('Age_Group_encoded')

print("Correlations with Age_Group for the Full Dataset (df):")
display(df_correlations.sort_values(ascending=False).head(10))
display(df_correlations.sort_values(ascending=True).head(10))

## Feature Selection: Top 25% Correlated Features with 'Age_Group'

In [ ]:
import numpy as np

# Ensure df_corr is updated in case there were changes to df since its last creation
df_corr = df.copy()
df_corr['Age_Group_encoded'] = le.transform(df_corr['Age_Group'])

# Drop the original categorical 'Age_Group' column before calculating correlations
df_corr = df_corr.drop(columns=['Age_Group'])

# Calculate absolute correlations with 'Age_Group_encoded'
absolute_correlations = df_corr.corr()['Age_Group_encoded'].abs().drop('Age_Group_encoded')

# Determine the number of top features (25%)
num_features_to_keep = int(len(absolute_correlations) * 0.25)

# Get the names of the top features
top_features = absolute_correlations.nlargest(num_features_to_keep).index.tolist()

print(f"Original number of features: {len(df_corr.columns) - 1}")
print(f"Number of features to keep (top 25%): {num_features_to_keep}")
print("Top features selected:")
display(top_features)

# Filter the original df to keep only these top features and the 'Age_Group' column
df = df[top_features + ['Age_Group']]

print("\nDataFrame 'df' after filtering by top correlated features:")
display(df.head())

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(df, test_size=0.7, random_state=42, stratify=df['Age_Group'])

print(f"Shape of training set (df_train): {df_train.shape}")
print(f"Shape of validation set (df_val): {df_val.shape}")

### Separate features (X) and target (y)

In [ ]:
TARGET_COLUMN = 'Age_Group'

X_train = df_train.drop(columns=[TARGET_COLUMN])
y_train = df_train[TARGET_COLUMN]

X_val = df_val.drop(columns=[TARGET_COLUMN])
y_val = df_val[TARGET_COLUMN]

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")

In [ ]:
X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy()


In [ ]:


# 1. Xác định danh sách cột phân loại (categorical) nếu có.
# Nhập index của các cột phân loại trong X_train. Ví dụ: categorical_cols = (1, 3)
# Nếu tất cả các cột của bạn đều là dữ liệu số liên tục, hãy để tuple rỗng:
categorical_columns = ()
num_classes = 5
# 2. Khởi tạo đối tượng ctdGAN với các siêu tham số phù hợp
gan = ctd_gan.ctdGAN(
    discriminator=(256, 256),   # Kiến trúc mạng Critic (Discriminator)
    generator=(256, 256),       # Kiến trúc mạng Generator
    epochs=150,                 # Số chu kỳ huấn luyện đối kháng (khuyên dùng từ 100-300 epochs để tối ưu chất lượng)
    batch_size=100,             # Kích thước lô huấn luyện
    pac=10,                     # Đóng gói mẫu chống sụp đổ chế độ (mode collapse)
    embedding_dim=128,          # Số chiều của không gian ẩn (latent space)
    max_clusters=10,            # Số phân cụm tối đa trong không gian con thực tế
    cluster_method='kmeans',    # Thuật toán phân cụm không gian thực
    scaler='mms11',             # Kỹ thuật chuẩn hóa đưa dữ liệu về khoảng [-1, 1]
    sampling_strategy='create-new', # Chiến lược sinh mẫu điều kiện
    random_state=42             # Seed để đảm bảo kết quả có thể tái lặp
)

In [ ]:


# Encode y_train_np to numerical values

y_train_encoded_np = le.fit_transform(y_train_np)

# 3. Huấn luyện mô hình và thực hiện sinh thêm mẫu (Oversampling)
# Hàm này sẽ học phân phối dữ liệu gốc và tự động sinh thêm dữ liệu mới
gan.fit(X_train_np, y_train_encoded_np)

In [ ]:
for index, label in enumerate(le.classes_):
    print(f"{index} -> {label}")

In [ ]:
z_gen = gan.sample(5000,0)

In [ ]:
fi_gen = gan.sample(5000,1)

In [ ]:
s_gen = gan.sample(5000,2)

In [ ]:
t_gen = gan.sample(5000,3)

In [ ]:
fo_gen = gan.sample(5000,4)

In [32]:
import numpy as np
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import rbf_kernel

# =====================================================================
# STEP 1: ĐỒNG BỘ HÓA DỮ LIỆU THẬT & GỘP NHÃN DỮ LIỆU TẠO RA (AI SYNTH)
# =====================================================================

# 1. Khôi phục nhãn chuỗi gốc cho dữ liệu giả từ chính cấu trúc mảng sinh của bạn
labels_z_synth = np.full(z_gen.shape[0], '10-19')
labels_fi_synth = np.full(fi_gen.shape[0], '20-29')
labels_s_synth = np.full(s_gen.shape[0], '30-39')
labels_t_synth = np.full(t_gen.shape[0], '40-49')
labels_fo_synth = np.full(fo_gen.shape[0], '50-59')

# Gộp ma trận đặc trưng (X) và mảng nhãn chuỗi (y) của tập dữ liệu ctdGAN
X_synth = np.vstack((z_gen,fi_gen, s_gen, t_gen, fo_gen))
y_synth_str = np.concatenate((labels_z_synth, labels_fi_synth,labels_s_synth, labels_t_synth,labels_fo_synth))

# 2. Đồng bộ hóa nhãn số nguyên bằng đối tượng LabelEncoder (le) bạn đã dùng cho dữ liệu thật
# Điều này giúp y_synth_encoded tương thích hoàn toàn với y_train_encoded_np khi huấn luyện Downstream ML
y_synth_encoded = le.transform(y_synth_str)

# Ánh xạ các mảng dữ liệu gốc của bạn vào luồng xử lý chính
X_real = X_train_np
y_real_encoded = y_train_encoded_np

print("--- THÔNG SỐ ĐẦU VÀO HỆ THỐNG ---")
print(f" -> Kích thước tập dữ liệu THẬT (X_train_np): {X_real.shape}")
print(f" -> Kích thước tập dữ liệu GIẢ   (X_synth):    {X_synth.shape}")


# =====================================================================
# STEP 2: THUẬT TOÁN KIỂM ĐỊNH PHÂN PHỐI ĐÃ KHẮC PHỤC LỖI QUY MÔ (DATA SCALE)
# =====================================================================

def calculate_corrected_synos_metrics(X_real_np, X_synth_np):
    """
    Hàm tính toán khoảng cách phân phối đồng bộ thang đo.
    Sử dụng StandardScaler của dữ liệu THẬT để đưa cả hai tập về cùng phân phối (Mean=0, Std=1).
    """
    # 1. Tiến hành Z-score Scaling để triệt tiêu ảnh hưởng đơn vị đo thô (mm)
    scaler = StandardScaler()
    X_real_scaled = scaler.fit_transform(X_real_np)
    X_synth_scaled = scaler.transform(X_synth_np)

    # 2. Đo lường Wasserstein Distance đơn biến trên dữ liệu đã Chuẩn hóa
    num_features = X_real_scaled.shape[1]
    wd_scaled_list = []

    for i in range(num_features):
        wd = wasserstein_distance(X_real_scaled[:, i], X_synth_scaled[:, i])
        wd_scaled_list.append(wd)

    mean_wd_scaled = np.mean(wd_scaled_list)
    # Quy đổi khoảng cách chuẩn hóa sang tỉ lệ % độ tin cậy hình thái cột
    wd_percentage = np.exp(-mean_wd_scaled) * 100

    # 3. Đo lường MMD đa biến trên không gian phân phối gốc (RBF Kernel tự kiểm soát biên độ)
    gamma = 1.0 / X_real_np.shape[1]
    K_real_real = rbf_kernel(X_real_np, X_real_np, gamma=gamma)
    K_synth_synth = rbf_kernel(X_synth_np, X_synth_np, gamma=gamma)
    K_real_synth = rbf_kernel(X_real_np, X_synth_np, gamma=gamma)

    mmd_squared = K_real_real.mean() + K_synth_synth.mean() - 2 * K_real_synth.mean()
    raw_mmd = np.sqrt(max(0, mmd_squared))
    # Quy đổi khoảng cách đa biến sang tỉ lệ % độ tin cậy tương quan giải phẫu
    mmd_percentage = np.exp(-raw_mmd) * 100

    # 4. Tính điểm số tin cậy phân phối tổng hợp (Overall Score)
    overall_trust_score = (wd_percentage + mmd_percentage) / 2

    return mean_wd_scaled, wd_percentage, raw_mmd, mmd_percentage, overall_trust_score


# =====================================================================
# STEP 3: THỰC THI CHẤM ĐIỂM & ĐÓNG GÓI KẾT QUẢ BÁO CÁO NGHIÊN CỨU
# =====================================================================

# Chạy thuật toán xử lý đồng bộ
raw_wd, wd_pct, raw_mmd, mmd_pct, overall_score = calculate_corrected_synos_metrics(X_real, X_synth)

# Xuất bảng hiển thị kết quả chuẩn mực khoa học
print("\n" + "="*55)
print("   KẾT QUẢ KIỂM ĐỊNH PHÂN PHỐI DỮ LIỆU TỔNG HỢP HOÀN CHỈNH")
print("="*55)
print(f"1. Wasserstein Distance Metric (Đơn biến - Đã chuẩn hóa):")
print(f"   - Khoảng cách thô định lượng: {raw_wd:.4f}")
print(f"   - Độ tin cậy hình thái cột   : {wd_pct:.2f}%")
print(f"     (Tránh lỗi tràn số mũ của dữ liệu trắc lượng thô)\n")
print(f"2. Maximum Mean Discrepancy Metric (Đa biến toàn cục):")
print(f"   - Khoảng cách thô tương quan: {raw_mmd:.4f}")
print(f"   - Độ tin cậy tương quan giải phẫu (Allometry): {mmd_pct:.2f}%\n")
print("-" * 55)
print(f"--> OVERALL DISTRIBUTION TRUST SCORE: {overall_score:.2f}%")
print("="*55)

--- THÔNG SỐ ĐẦU VÀO HỆ THỐNG ---
 -> Kích thước tập dữ liệu THẬT (X_train_np): (1820, 23)
 -> Kích thước tập dữ liệu GIẢ   (X_synth):    (25000, 23)

   KẾT QUẢ KIỂM ĐỊNH PHÂN PHỐI DỮ LIỆU TỔNG HỢP HOÀN CHỈNH
1. Wasserstein Distance Metric (Đơn biến - Đã chuẩn hóa):
   - Khoảng cách thô định lượng: 0.1826
   - Độ tin cậy hình thái cột   : 83.31%
     (Tránh lỗi tràn số mũ của dữ liệu trắc lượng thô)

2. Maximum Mean Discrepancy Metric (Đa biến toàn cục):
   - Khoảng cách thô tương quan: 0.0243
   - Độ tin cậy tương quan giải phẫu (Allometry): 97.60%

-------------------------------------------------------
--> OVERALL DISTRIBUTION TRUST SCORE: 90.46%


In [ ]:
!pip install sdmetrics

In [ ]:
from sdmetrics.reports.single_table import QualityReport

In [35]:
num_features = X_train_np.shape[1]
column_names = [f'feature_{i}' for i in range(num_features)]

# 1. Khởi tạo DataFrame cho dữ liệu THẬT (Gồm các đặc trưng xương + cột nhãn giới tính số nguyên)
real_df = pd.DataFrame(X_train_np, columns=column_names)
real_df['Age_Group'] = y_train_encoded_np



synth_df = pd.DataFrame(X_synth, columns=column_names)
synth_df['Age_Group'] = y_synth_encoded


# =====================================================================
# STEP 2: KHỞI TẠO METADATA THEO TIÊU CHUẨN SDMETRICS
# =====================================================================
# SDMetrics cần biết cột nào là số liên tục (numerical) và cột nào là phân loại (categorical)
metadata = {
    'columns': {col: {'sdtype': 'numerical'} for col in column_names},
}
# Bổ sung thông tin cột gender là biến phân loại (nhãn encoded số nguyên)
metadata['columns']['Age_Group'] = {'sdtype': 'categorical'}


# =====================================================================
# STEP 3: CHẠY BÁO CÁO CHẤT LƯỢNG VÀ TRÍCH XUẤT SỐ % ĐỘ TIN CẬY
# =====================================================================
print("--- ĐANG TIẾN HÀNH CHẤM ĐIỂM VỚI SDMETRICS ---")

# Khởi tạo và tạo báo cáo
report = QualityReport()
report.generate(real_df, synth_df, metadata)

# 1. Lấy điểm số phần trăm tổng quát (Overall Score)
overall_score_pct = report.get_score() * 100

# 2. Lấy điểm số chi tiết theo từng khía cạnh cốt lõi
properties_df = report.get_properties()

print("\n" + "="*55)
print("   BÁO CÁO CHẤT LƯỢNG DỮ LIỆU TỔNG HỢP TỪ SDMETRICS")
print("="*55)
for index, row in properties_df.iterrows():
    property_name = row['Property']
    score_pct = row['Score'] * 100

    # Dịch nghĩa các thuộc tính của SDMetrics sang ngôn ngữ luận án SYNOS
    if property_name == 'Column Shapes':
        print(f"1. Độ tương đồng Hình dáng Đơn biến (Column Shapes): {score_pct:.2f}%")
        print("   (Bao gồm kiểm định KS-Test cho biến số và Chi-Square cho giới tính)")
    elif property_name == 'Column Pair Trends':
        print(f"\n2. Độ tương đồng Tương quan Đa biến (Column Pair Trends): {score_pct:.2f}%")
        print("   (Kiểm tra ma trận tương quan chéo giữ luật giải phẫu Allometry)")

print("-" * 55)
print(f"--> OVERALL SDMETRICS TRUST SCORE: {overall_score_pct:.2f}%")
print("="*55)

--- ĐANG TIẾN HÀNH CHẤM ĐIỂM VỚI SDMETRICS ---
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 24/24 [00:00<00:00, 253.80it/s]|
Column Shapes Score: 88.96%

(2/2) Evaluating Column Pair Trends: |██████████| 276/276 [00:04<00:00, 61.33it/s]|
Column Pair Trends Score: 97.89%

Overall Score (Average): 93.42%


   BÁO CÁO CHẤT LƯỢNG DỮ LIỆU TỔNG HỢP TỪ SDMETRICS
1. Độ tương đồng Hình dáng Đơn biến (Column Shapes): 88.96%
   (Bao gồm kiểm định KS-Test cho biến số và Chi-Square cho giới tính)

2. Độ tương đồng Tương quan Đa biến (Column Pair Trends): 97.89%
   (Kiểm tra ma trận tương quan chéo giữ luật giải phẫu Allometry)
-------------------------------------------------------
--> OVERALL SDMETRICS TRUST SCORE: 93.42%


In [ ]:
synth_df.head()

In [37]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np # Import numpy for nan

# =====================================================================
# DATA PREPROCESSING & DATASET SYNCHRONISATION
# =====================================================================
target_col = TARGET_COLUMN # Use the global TARGET_COLUMN which is 'Age_Group'

# Real data (X_train_np and y_train_encoded_np are already prepared)
X_train_real = X_train_np
y_train_real = y_train_encoded_np

# For validation data, X_val is a DataFrame, and y_val needs to be encoded
X_val_real = X_val.values # Convert X_val DataFrame to numpy array
y_val_encoded_np = le.transform(y_val.to_numpy()) # Encode y_val Series to numpy array
y_val_real = y_val_encoded_np

# Synthetic data (X_synth and y_synth_encoded are already prepared)
X_train_synth = X_synth
y_train_synth = y_synth_encoded


# =====================================================================
# MODEL ARCHITECTURE CONFIGURATION
# =====================================================================
models_config = {
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=5, random_state=42, eval_metric='mlogloss'), # Use mlogloss for multi-class
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42) # Ensure probability=True for AUC
}

# =====================================================================
# EVALUATION EXECUTION (Baseline TRTR vs. Generative TSTR)
# =====================================================================
report_data = []

def evaluate_pipeline(model, X_train, y_train, X_val):
    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    # For multi-class, predict_proba returns array of shape (n_samples, n_classes)
    # roc_auc_score needs to handle multi-class. `multi_class='ovr'` and `average='macro'` are suitable here.
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_val)
    else:
        probs = None # If model does not support predict_proba, AUC will be NaN

    return preds, probs

for name, model in models_config.items():
    # 1. Baseline Configuration (Train Real, Test Real)
    y_pred_real, y_prob_real = evaluate_pipeline(model, X_train_real, y_train_real, X_val_real)

    # 2. Synthetic Configuration (Train Synthetic, Test Real)
    y_pred_synth, y_prob_synth = evaluate_pipeline(model, X_train_synth, y_train_synth, X_val_real)

    current_report = {
        "Model": name,
        "Acc_Real": accuracy_score(y_val_real, y_pred_real),
        "F1_Real": f1_score(y_val_real, y_pred_real, average='macro'),
        "Acc_Synth": accuracy_score(y_val_real, y_pred_synth),
        "F1_Synth": f1_score(y_val_real, y_pred_synth, average='macro'),
    }

    # Handle AUC for multi-class
    if y_prob_real is not None and y_prob_real.ndim == 2: # Check if it's multi-class probability array
        current_report["AUC_Real"] = roc_auc_score(y_val_real, y_prob_real, multi_class='ovr', average='macro')
    else:
        current_report["AUC_Real"] = np.nan # Placeholder if AUC cannot be computed

    if y_prob_synth is not None and y_prob_synth.ndim == 2:
        current_report["AUC_Synth"] = roc_auc_score(y_val_real, y_prob_synth, multi_class='ovr', average='macro')
    else:
        current_report["AUC_Synth"] = np.nan # Placeholder if AUC cannot be computed

    report_data.append(current_report)


# =====================================================================
# PERFORMANCE COMPILATION & GENERATIVE EFFICACY ANALYSIS
# =====================================================================
df_res = pd.DataFrame(report_data).set_index("Model")

for metric in ['Acc', 'F1', 'AUC']:
    # Only calculate efficacy if Real metric is not 0 to avoid division by zero
    df_res[f'{metric}_Efficacy (%)'] = (df_res[f'{metric}_Synth'] / df_res[f'{metric}_Real']).replace([np.inf, -np.inf], np.nan) * 100

print("\n" + "="*85)
print("             BẢNG TỔNG HỢP HIỆU NĂNG ĐỘ TIN CẬY (TSTR vs TRTR)")
print("="*85)

format_dict = {col: '{:,.2f}%'.format for col in df_res.columns if 'Efficacy' in col}
other_cols = {col: '{:,.4f}'.format for col in df_res.columns if 'Efficacy' not in col}

# Use a custom function for formatting to handle NaN values gracefully
def format_values(val):
    if pd.isna(val):
        return '-'
    return '{:,.4f}'.format(val)

print(df_res.style.format({**format_dict, **{col: format_values for col in df_res.columns if 'Efficacy' not in col}}).to_string())
print("="*85)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c


             BẢNG TỔNG HỢP HIỆU NĂNG ĐỘ TIN CẬY (TSTR vs TRTR)
 Acc_Real F1_Real Acc_Synth F1_Synth AUC_Real AUC_Synth Acc_Efficacy (%) F1_Efficacy (%) AUC_Efficacy (%)
Model         
XGBoost 0.5311 0.2701 0.2500 0.2136 0.6997 0.6163 47.07% 79.07% 88.08%
Random Forest 0.5341 0.1851 0.1749 0.1642 0.7152 0.6642 32.75% 88.69% 92.86%
Logistic Regression 0.5523 0.2759 0.2665 0.2239 0.7225 0.6018 48.25% 81.15% 83.30%
SVM 0.5214 0.1371 0.2041 0.1869 0.7337 0.6465 39.14% 136.32% 88.12%



In [ ]:
df_res

In [ ]:
# Kiểm tra xem dữ liệu giả có bị trùng với dữ liệu thật không
# Sử dụng real_df thay vì df_train vì real_df có cùng cấu trúc cột và kiểu dữ liệu (đặc biệt là 'Gender') với synth_df
duplicates = pd.merge(synth_df, real_df, how='inner')
print(f"Số lượng bản ghi bị sao chép y hệt: {len(duplicates)}")

In [ ]:
# 1. Kiểm tra sự trùng lặp giữa Train và Val
intersection = pd.merge(df_train, df_val, how='inner')
print(f"Số lượng bản ghi trùng lặp giữa Train và Val: {len(intersection)}")

# 2. Kiểm tra các cột có tính chất định danh
# Tìm các cột có số lượng giá trị duy nhất cao bằng số lượng mẫu
for col in df_train.columns:
    if df_train[col].nunique() == len(df_train):
        print(f"Cảnh báo: Cột '{col}' có thể là biến ID gây rò rỉ dữ liệu!")